# Essentiality calibration -- the CRISPR common-essential gate

**Why we're doing this.** `CONSTRAINTS.md` S1: the dependency layer scores every gene's Chronos value identically, but a pan-essential gene (most cell lines die without it, regardless of the query) produces a strongly negative Chronos score in nearly every line -- "this line dies without the gene" then reads as strong, selective inclusion evidence when it is really "every line dies without this gene", biologically vacuous. `METHOD_DECISION.md` SS2 calls a never-silent warning here mandatory. `eda/single_layer/15` SS2-3 already looked at this distribution informally; this notebook ports the *rule* into a reproducible, versioned resource the scorer actually loads.

**The computation lives in `scoring/build_essentiality_constants.py`, not in this notebook.** That script is what actually ships the constants (`python scoring/build_essentiality_constants.py`, run by hand from the repo root -- or `python scoring/build_calibration_constants.py` to rebuild every calibration file in one command). This notebook imports the very same functions and walks through them step by step, so the audit you are reading and the file the scorer loads can never drift apart.

**Output** -- `scoring/resources/common_essential_genes.json` -- a static, per-gene property computed once over the full `data/processed/dependency.csv` (21.4M rows), independent of any query, exactly like the RNA/protein/dependency/copy-number L/T constants already are. Loaded by `scoring/desirability.py::load_essentiality_constants`.

**The rule, stated so it can be challenged.** Per gene, over every line with >=30 measured Chronos profiles: `fraction_strong = ` the share of lines with `dependency_score < -1.0` ("strongly negative" -- the same cutoff `eda/single_layer/15` SS2 already uses to describe "strong dependency", where 5.12% of all gene x model cells fall below it). A gene is flagged `pan_essential` if `fraction_strong >= 0.90`.

**Honesty about status.** Both numbers (`-1.0`, `0.90`) are `EDA-inherited`, not `cited` -- Hart et al. (2014/2017) (the core-essential gold standard) and Dempster et al. (2021) (the Chronos method itself) are the closest available published references, but neither paper specifies this exact fraction-of-lines rule or these exact numbers on the Chronos scale. `docs/plan/PARAMETERS.md` row 40 records this precisely, and `docs/plan/EXECUTE.md` V6-3 schedules both for the sensitivity sweep. This is a defensible default, not a final calibration.

In [ ]:
import sys
import time

sys.path.insert(0, "..")  # this notebook runs from scoring/notebooks/; the builder is one level up

import build_essentiality_constants as build

DATA_DIR = "../../data/processed"
RESOURCES_DIR = "../resources"

STRONG_DEPENDENCY_CUTOFF = build.STRONG_DEPENDENCY_CUTOFF
PAN_ESSENTIAL_FRACTION_THRESHOLD = build.PAN_ESSENTIAL_FRACTION_THRESHOLD
MIN_N = build.MIN_N
print(f"cutoff={STRONG_DEPENDENCY_CUTOFF}, threshold={PAN_ESSENTIAL_FRACTION_THRESHOLD}, MIN_N={MIN_N}")

## 1. Load `dependency.csv` and compute per-gene fraction-strongly-dependent

In [ ]:
t0 = time.time()
dependency = build.load_dependency(DATA_DIR)
print(f"loaded {len(dependency):,} rows ({time.time() - t0:.1f}s)")

grouped = build.compute_fraction_strong(dependency, STRONG_DEPENDENCY_CUTOFF, MIN_N)
print(f"n_genes_total (>= {MIN_N} measured lines) = {len(grouped):,}")

## 2. Flag pan-essential genes and sanity-check the top hits

In [ ]:
pan_essential = build.flag_pan_essential(grouped, PAN_ESSENTIAL_FRACTION_THRESHOLD)
print(f"n_genes_flagged pan_essential = {len(pan_essential):,}")

# sanity check: the strongest hits should be recognisable core-machinery genes (ribosomal
# proteins, spliceosome components), not an arbitrary-looking set -- a quick biological
# plausibility check on the rule, not a formal validation.
top_hits = build.sanity_check_top_hits(pan_essential, DATA_DIR)
print(top_hits[["ensembl_id", "symbol", "fraction_strong"]].to_string(index=False))

## 3. Write `scoring/resources/common_essential_genes.json`

In [ ]:
import json

constants = build.build_constants(grouped, pan_essential)

out_path = f"{RESOURCES_DIR}/common_essential_genes.json"
with open(out_path, "w") as f:
    json.dump(constants, f, indent=2)
print(f"Wrote {out_path}")